# Extract AIS data

Reads the monthly NOAA MarineCadastre zip files from a folder and keeps the reports inside a bounding box.
Download the zips first: `python scripts/download_noaa.py --start 2021-01-01 --end 2021-01-31 --region hawaii_2021_01 --keep-zips`.
That script already filters the bounding box and writes `data/hawaii_2021_01.parquet`; this notebook shows how to do it from the zips.

Figures and maps are also saved to `outputs/` (PNG for matplotlib, HTML for interactive maps).

In [ ]:
from pathlib import Path

import folium

from ais_anomaly.io import read_zip_folder, save_figure
from ais_anomaly.maps import BASEMAP

DATA = Path("../data")

In [ ]:
# Hawaii bounding box
min_lat, max_lat, min_lon, max_lon = 18, 22.5, -160, -150
df_filtered_combined = read_zip_folder(DATA / "raw" / "hawaii_2021_01", min_lat, max_lat, min_lon, max_lon)

In [ ]:
t = df_filtered_combined["BaseDateTime"]
print(f"Reports: {len(df_filtered_combined)}  period: {t.iloc[0]} - {t.iloc[-1]}")
df_filtered_combined.head()

In [ ]:
# Map centred on Hawaii, one marker every 10000 reports
m = folium.Map(location=[21, -158], tiles=BASEMAP, zoom_start=9)
for _, row in df_filtered_combined.iloc[::10000].iterrows():
    folium.Marker(location=[row["LAT"], row["LON"]]).add_to(m)
save_figure(m, "01_ais_overview_map")
m